## S2.3 — DAG (Directed Acyclic Graph)
**Date completed:** May 2025  
**Status:** Complete  
**Interview covered:** Q21 — What is DAG in Spark?

## What is a DAG?

DAG = Directed Acyclic Graph

| Word | Meaning | Spark example |
|------|---------|---------------|
| Graph | Boxes connected by arrows | filter → groupBy → count |
| Directed | Arrows go ONE WAY only | Bronze → Silver → Gold (never reverse) |
| Acyclic | No loops, no cycles | Cannot return to previous step |

## Medallion Architecture = a DAG

In [0]:
## How Spark builds a DAG
# python
# Each transformation adds a node to the DAG (lazy - no execution)

# Each transformation adds a node to the DAG (lazy - no execution)
df = spark.range(1000000)        # Node 1: Range
df = df.filter(df.id > 500000)   # Node 2: Filter
df = df.groupBy("partition_id")  # Node 3: GroupBy

result = df.count()              # ACTION → DAG executes NOW

# DAG built: Range → Filter → GroupBy → Count

## Why Spark uses a DAG
"""
Spark builds the full plan BEFORE executing
Catalyst Optimizer reorders steps for efficiency
Example: filter early → fewer rows for groupBy → faster
If any node fails → all downstream nodes stop (like Bronze failing)
"""

## Key Interview Answer — Q21
# A DAG in Spark is the execution plan built from transformations.
"""
It is directed (flows one way), acyclic (no loops), and allows
Catalyst Optimizer to reorder operations for maximum efficiency
before any data is processed.
"""

# What DAG solves — 4 benefits:

####  **Optimization before touching data:**

DAG sees your FULL plan before executing
Moves filter before select automatically
Reads only needed columns
You never wrote this optimization — Spark did it

## Fault tolerance:
Node 3 fails during execution
Spark knows the DAG
Spark recomputes ONLY from Node 3
Does not restart from Node 1
= faster recovery in production

## Parallel execution:
DAG shows which nodes have no dependency
Those run in PARALLEL automatically
Without DAG — everything runs sequentially

## Lineage:
DAG records HOW every row was created
If data is lost — Spark recomputes using the DAG
This is called RDD lineage — we cover in S2.7